<a href="https://colab.research.google.com/github/Mohan102945/mohan102945.github.io/blob/main/LLM_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import urllib.request

In [31]:
url = ("https://raw.githubusercontent.com/rasbt/"
"LLMs-from-scratch/main/ch02/01_main-chapter-code/"
"the-verdict.txt")

file_path = "the-verdict.txt"

In [32]:
urllib.request.urlretrieve(url,file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7d59a51db9e0>)

In [33]:
with open(file_path,"r",encoding="utf-8") as f:
    raw_text = f.read()
print(raw_text[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [34]:
import re
text = "Hello, world, This is a test."
token = re.split(r'(\s)',text)
print(token)

['Hello,', ' ', 'world,', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [35]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', ',', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [36]:
token = [item for item in token if item.strip()]
print(token)

['Hello,', 'world,', 'This', 'is', 'a', 'test.']


In [37]:
text = "Hello, world. Is this-- a test?"
token = re.split(r'([,.:;?_!"()\']|--|\s)',text)
print(token)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'Is', ' ', 'this', '--', '', ' ', 'a', ' ', 'test', '?', '']


In [38]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
print(preprocessed[:100])

['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ', 'Gisburn', ' ', 'rather', ' ', 'a', ' ', 'cheap', ' ', 'genius', '--', 'though', ' ', 'a', ' ', 'good', ' ', 'fellow', ' ', 'enough', '--', 'so', ' ', 'it', ' ', 'was', ' ', 'no', ' ', 'great', ' ', 'surprise', ' ', 'to', ' ', 'me', ' ', 'to', ' ', 'hear', ' ', 'that', ',', '', ' ', 'in', ' ', 'the', ' ', 'height', ' ', 'of', ' ', 'his', ' ', 'glory', ',', '', ' ', 'he', ' ', 'had', ' ', 'dropped', ' ', 'his', ' ', 'painting', ',', '', ' ', 'married', ' ', 'a', ' ', 'rich', ' ', 'widow', ',', '', ' ', 'and', ' ', 'established', ' ', 'himself', ' ', 'in', ' ', 'a', ' ']


In [39]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1133


In [40]:
vocab = {word:index for index, word in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
  print(item)
  if i >= 50:
    break

('', 0)
('\n', 1)
(' ', 2)
('!', 3)
('"', 4)
("'", 5)
('(', 6)
(')', 7)
(',', 8)
('--', 9)
('.', 10)
(':', 11)
(';', 12)
('?', 13)
('A', 14)
('Ah', 15)
('Among', 16)
('And', 17)
('Are', 18)
('Arrt', 19)
('As', 20)
('At', 21)
('Be', 22)
('Begin', 23)
('Burlington', 24)
('But', 25)
('By', 26)
('Carlo', 27)
('Chicago', 28)
('Claude', 29)
('Come', 30)
('Croft', 31)
('Destroyed', 32)
('Devonshire', 33)
('Don', 34)
('Dubarry', 35)
('Emperors', 36)
('Florence', 37)
('For', 38)
('Gallery', 39)
('Gideon', 40)
('Gisburn', 41)
('Gisburns', 42)
('Grafton', 43)
('Greek', 44)
('Grindle', 45)
('Grindles', 46)
('HAD', 47)
('Had', 48)
('Hang', 49)
('Has', 50)


In [41]:
class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s,i in vocab.items()}

  def encode(self,text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids = [self.str_to_int[item] for item in preprocessed]
    return ids

  def decode(self,ids):
    text = [self.int_to_str[item] for item in ids]
    return " ".join(text)

In [42]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[4, 59, 5, 853, 991, 605, 536, 749, 8, 1129, 599, 8, 4, 70, 10, 41, 854, 1111, 757, 796, 10]


In [43]:
print(tokenizer.decode(ids))

" It ' s the last he painted , you know , " Mrs . Gisburn said with pardonable pride .


### Adding special context tokens - <|unk|>, <|endoftext|> to the vocabulary

In [44]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab = {item:i for i,item in enumerate(all_tokens)}
print(len(vocab.items()))

1135


In [45]:
for i,item in enumerate(list(vocab.items())[-5:]):
  print(item)

('younger', 1130)
('your', 1131)
('yourself', 1132)
('<|endoftext|>', 1133)
('<|unk|>', 1134)


In [46]:
class SimpleTokenizerV2:
  def __init__(self,vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s,i in vocab.items()}

  def encode(self,text):
    preprocessed = re.split(r'([,.!"\'()?!:;_\[\]]|--|\s)',text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
    ids = [self.str_to_int[item] for item in preprocessed]
    return ids

  def decode(self,ids):
    text = " ".join([self.int_to_str[item] for item in ids])
    text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
    return text

In [47]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = "<|endoftext|>".join([text1,text2])
print(text)

Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.


In [48]:
tokenizer = SimpleTokenizerV2(vocab)

In [49]:
print(tokenizer.encode(text))

[1134, 8, 358, 1129, 631, 978, 13, 1134, 991, 959, 987, 725, 991, 1134, 10]


In [50]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|unk|> the sunlit terraces of the <|unk|>.


# Byte Pain Encoding

In [51]:
import tiktoken

In [52]:
from importlib.metadata import version
print(version("tiktoken"))

0.12.0


In [53]:
tokenizer = tiktoken.get_encoding("gpt2")
integer = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integer)

[15496, 11, 466, 345, 588, 8887, 30, 50256, 818, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13]


In [54]:
print(tokenizer.decode(integer))

Hello, do you like tea?<|endoftext|>In the sunlit terraces of the palace.


In [55]:
print(tokenizer.encode("Akwirw ier"))

[33901, 86, 343, 86, 220, 959]


In [56]:
class BPETokenizerSimple:
  def __init__(self):
    self.vocab = {}
    self.inverse_vocab = {}
    self.bpe_merges = {}
    self.bpe_ranks = {}

  def train(self,text,vocab_size,allowed_specials = {"<|endoftext|>"}):
    processed_text = []
    for i,char in enumerate(text):
      if char == " " and i != 0:
        processed_text.append("Ġ")
      if char != " ":
        processed_text.append(char)
    processed_text = "".join(processed_text)

    unique_chars = [chr(i) for i in range(256)]
    unique_chars.extend([char for char in sorted(set(processed_text)) if char not in unique_chars])
    if "Ġ" not in unique_chars:
      unique_chars.append("Ġ")

    self.vocab = {i:char for i, char in enumerate(unique_chars)}
    self.inverse_vocab = {char:i for char, i in self.vocab.items()}

    if allowed_special:
      for char in allowed_special:
        if char not in self.inverse_vocab:
           self.vocab[len(self.vocab)] = char
           self.inverse_vocab[char] = len(self.vocab)

    token_ids = [self.vocab(char) for char in processed_text]

    for new_id in range(len(self.vocab,vocab_size)):
      pair_id = self.find_freq_pair(token_ids, mode = "most")
      if pair_id is None:
        break
      token_ids = self.replace_pair(token_ids,pair_id,new_id)
      self.bpe_merges[pair_id] = new_id



# Dataset and DataLoader

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
  def __init__(self,text,tokenizer,max_length,stride):
    self.input_ids = []
    self.output_ids = []

    token_ids = tokenizer.encode(text)

    for i in range (0, len(token_ids)-max_length,stride):
      input_chunk = token_ids[i:i+max_length]
      output_chunk = token_ids[i+1:i+1+max_length]
      self.input_ids.append(torch.tensor(input_chunk))
      self.output_ids.append(torch.tensor(output_chunk))
  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self,idx):
    return self.input_ids[idx],self.output_ids[idx]